# Kalshi vs. Polymarket — Full 9-Event Pull (fixed version)

This notebook downloads price history for all 9 matched events, lines up
Kalshi and Polymarket prices in time, computes the gap
`G(t) = Kalshi − Polymarket` for each event, flags outlier points, saves a CSV
per event, and plots prices + gap.

Output layout is the same as before:

```
out/summary.csv            one row per event
out/<event>/gap.csv        aligned prices and the gap, with outlier flags
out/<event>/gap_plot.png   prices on top, gap below
```

### What changed from the previous version

| # | Problem in the old notebook | Fix |
|---|---|---|
| 1 | At settlement Kalshi's book goes to bid 0 / ask 1, so the midpoint is exactly 0.50. Those rows produced the ~0.5 "bad candles". | Each series is cut at the earlier of the two markets' close times, and Kalshi bars whose bid/ask spread is wider than `MAX_KALSHI_SPREAD` are dropped. |
| 2 | Kalshi and Polymarket prices were not clearly the same kind of price. | Both sides are quote midpoints (Polymarket's history is mid-based; its half-tick values like 0.875 show this). The Kalshi last-trade price is saved too (`kalshi_last`) for a sensitivity check. |
| 3 | `_as_probability` guessed units (`x > 1 → cents`), so a 1¢ quote sent in cents became $1.00. | Units are read from the field name / type, never guessed. Anything outside [0, 1] stops the run. |
| 4 | Prices were not range-checked (one Polymarket price was 1.0045). | Every price is checked to be in [0, 1]; bad points are dropped and counted. |
| 5 | Hourly data is too coarse to measure how fast gaps close. | Default is **1-minute** bars (`BAR_MINUTES`). Set it to 60 to get the old resolution. |
| 6 | Matching used `nearest` within 45 min, which could pair a price with one from up to 45 min in the future, and one Kalshi point could be reused. | Each Kalshi bar is matched to at most one Polymarket point within ±half a bar (±30 s at 1-min bars). The offset is saved (`poly_offset_s`). |
| 7 | The outlier MAD became 0 on flat stretches (→ NA → never flagged), and the window counted rows, not time. | Time-based window, and the MAD has a floor of half a cent. |
| 8 | Download failures were silent (bare `except`, missing chunks only printed). | Requests retry, errors are raised or counted, and the summary reports failed chunks and the largest time gap. |
| 9 | (README) said Polymarket's `data-api` endpoint was used; it returns nothing for older markets. | The CLOB `prices-history` endpoint is used at `fidelity=1`, which gives 1-minute data for all 9 markets. |
| 10 | Polymarket token lookup fell back to the first token, which could be the "No" side. | Every token ID is checked against Polymarket's market list and must be the **Yes** outcome, or the event stops. |

**Just run all cells top to bottom.** No API keys are needed.

In [ ]:
# Install what we need (already present on Colab; harmless if so)
%pip install -q requests pandas matplotlib

## Settings

In [ ]:
import json
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone, timedelta
import matplotlib.pyplot as plt
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Bar size in minutes. Kalshi only allows 1, 60 or 1440.
# 1 = fine enough to see gaps close; 60 = the old notebook's resolution.
BAR_MINUTES = 1

# Kalshi bars with a bid/ask spread wider than this are dropped: the midpoint
# of a very wide (or empty, bid 0 / ask 1) book is not a real price.
# 0.10 matches the rule Polymarket uses for its own displayed price.
MAX_KALSHI_SPREAD = 0.10

# Outlier flag: compare each gap to the median of the points within
# OUTLIER_WINDOW around it, scaled by the MAD (with a floor of MAD_FLOOR so
# flat stretches don't divide by zero).
OUTLIER_WINDOW = "31min" if BAR_MINUTES == 1 else f"{11 * BAR_MINUTES}min"
OUTLIER_Z = 5.0
MAD_FLOOR = 0.005

OUT_DIR = Path("out")

# Kalshi public API hosts (no key needed). trading-api.kalshi.com now
# returns 401 without a key, so it is not used.
KALSHI_BASES = (
    "https://api.elections.kalshi.com/trade-api/v2",
    "https://external-api.kalshi.com/trade-api/v2",
)
KALSHI_ALLOWED_PERIODS_MINUTES = (1, 60, 1440)
KALSHI_MAX_CANDLES_PER_REQUEST = 5000

POLY_GAMMA_BASE = "https://gamma-api.polymarket.com"
POLY_CLOB_BASE = "https://clob.polymarket.com"
POLY_MAX_WINDOW_DAYS = 14   # the CLOB rejects windows much longer than 15 days

assert BAR_MINUTES in KALSHI_ALLOWED_PERIODS_MINUTES, "Kalshi only allows 1, 60 or 1440 minute bars"

## HTTP helpers

One shared session that retries on rate limits and server errors. Anything
else that goes wrong raises an error instead of being skipped silently.

In [ ]:
class FetchError(RuntimeError):
    pass


def _make_session():
    retry = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        raise_on_status=False,
    )
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


SESSION = _make_session()


def http_get_json(url, params=None, allow_404=False):
    # Returns parsed JSON, or None for a 404 when allow_404=True.
    # Raises FetchError for everything else that isn't a 200.
    try:
        r = SESSION.get(url, params=params, timeout=30)
    except requests.RequestException as e:
        raise FetchError(f"network error for {url}: {e}") from e
    if r.status_code == 404 and allow_404:
        return None
    if r.status_code != 200:
        raise FetchError(f"HTTP {r.status_code} for {url} params={params}: {r.text[:200]}")
    return r.json()


def kalshi_get(path, params=None, allow_404=False):
    # Try each Kalshi host. A 404 is a real answer (market not in this tier),
    # so it is returned straight away rather than retried on another host.
    last_err = None
    for base in KALSHI_BASES:
        try:
            return http_get_json(f"{base}{path}", params, allow_404=allow_404)
        except FetchError as e:
            last_err = e
    raise last_err


def to_utc(date_str):
    return datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)


def parse_time(value):
    # Parses API timestamps such as '2026-09-16T17:59:00Z' or '2026-09-16 21:40:27+00'.
    if not value:
        return None
    return pd.Timestamp(value).tz_convert("UTC") if pd.Timestamp(value).tzinfo else pd.Timestamp(value, tz="UTC")

## Look up and verify markets

`get_kalshi_market` finds which Kalshi tier (live or `/historical/`) a ticker is
in and returns its close time. `get_polymarket_market` checks that a token ID is
the **Yes** side of a real Polymarket market and returns its close time.

In [ ]:
def get_kalshi_market(ticker):
    # Returns (market dict, tier) where tier is 'live' or 'historical'.
    body = kalshi_get(f"/markets/{ticker}", allow_404=True)
    if body is not None:
        return body["market"], "live"
    body = kalshi_get(f"/historical/markets/{ticker}", allow_404=True)
    if body is not None:
        return body["market"], "historical"
    raise FetchError(f"Kalshi ticker {ticker} not found in the live or historical API")


def _parse_json_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except ValueError:
            return []
    return value if isinstance(value, list) else []


def get_polymarket_market(token_id):
    # Finds the Polymarket market that owns token_id and checks it is the Yes token.
    token_id = str(token_id)
    for params in ({"clob_token_ids": token_id, "closed": "true"},
                   {"clob_token_ids": token_id}):
        markets = http_get_json(f"{POLY_GAMMA_BASE}/markets", params)
        if markets:
            break
    else:
        raise FetchError(f"Polymarket token {token_id} not found")

    m = markets[0]
    tokens = [str(t) for t in _parse_json_list(m.get("clobTokenIds"))]
    outcomes = _parse_json_list(m.get("outcomes"))
    if token_id not in tokens or len(tokens) != len(outcomes):
        raise FetchError(f"Polymarket token {token_id}: could not match it to an outcome")
    outcome = outcomes[tokens.index(token_id)]
    if outcome != "Yes":
        raise FetchError(
            f"Polymarket token {token_id} is the '{outcome}' side of "
            f"'{m.get('question')}', not 'Yes'. Use the Yes token."
        )
    return m


def find_kalshi_markets(series_ticker):
    # Lists every market in a Kalshi series (live and historical) so you can
    # pick the exact ticker.
    rows, seen = [], set()
    for path in ("/markets", "/historical/markets"):
        cursor = None
        for _ in range(50):
            params = {"series_ticker": series_ticker, "limit": 1000}
            if cursor:
                params["cursor"] = cursor
            body = kalshi_get(path, params)
            for m in body.get("markets", []):
                if m.get("ticker") and m["ticker"] not in seen:
                    seen.add(m["ticker"])
                    rows.append({
                        "ticker": m["ticker"],
                        "title": m.get("title"),
                        "yes_sub_title": m.get("yes_sub_title"),
                        "status": m.get("status"),
                        "close_time": m.get("close_time"),
                        "result": m.get("result"),
                        "source": "historical" if "historical" in path else "live",
                    })
            cursor = body.get("cursor")
            if not cursor:
                break
    return pd.DataFrame(rows)


def find_polymarket_event(slug_or_query):
    # Looks up a Polymarket event by slug (or text search) and lists its
    # markets with the Yes / No token ids.
    attempts = [
        {"slug": slug_or_query, "closed": "true", "limit": 100},
        {"search": slug_or_query, "closed": "true", "limit": 100},
        {"slug": slug_or_query, "limit": 100},
        {"search": slug_or_query, "limit": 100},
    ]
    events = []
    for params in attempts:
        events = http_get_json(f"{POLY_GAMMA_BASE}/events", params)
        if events:
            break

    rows = []
    for ev in events or []:
        for m in ev.get("markets", []):
            tokens = _parse_json_list(m.get("clobTokenIds"))
            outcomes = _parse_json_list(m.get("outcomes"))
            by_outcome = dict(zip(outcomes, tokens)) if len(outcomes) == len(tokens) else {}
            rows.append({
                "event_slug": ev.get("slug"),
                "market_question": m.get("question"),
                # No fallback to tokens[0]: if the outcomes aren't labelled
                # Yes/No we'd rather see None than silently get the wrong side.
                "yes_token_id": by_outcome.get("Yes"),
                "no_token_id": by_outcome.get("No"),
                "closed": m.get("closed"),
                "closed_time": m.get("closedTime"),
            })
    return pd.DataFrame(rows)


def resolve_polymarket_yes_token(event_slug, market_contains):
    df = find_polymarket_event(event_slug)
    if df.empty:
        print(f"No Polymarket event found for '{event_slug}'")
        return None
    mask = df["market_question"].fillna("").str.contains(market_contains, case=False, regex=False)
    matches = df[mask & df["yes_token_id"].notna()]
    if len(matches) != 1:
        print(f"Found {len(matches)} matches:")
        print(df[mask][["market_question", "yes_token_id"]])
        return None
    return str(matches.iloc[0]["yes_token_id"])

## Download: Kalshi candles

Units are read from the field, not guessed:

* the live API sends `close_dollars` as a dollar string (`"0.8600"`);
* the historical API sends `close` as a dollar string (`"0.9900"`);
* an integer in a non-`_dollars` field is the old cents format (`86`).

The price used is the bid/ask **midpoint** at the end of each bar. Bars with a
spread wider than `MAX_KALSHI_SPREAD` (including the bid 0 / ask 1 book at
settlement) get no price and are dropped later.

In [ ]:
def _price_from_block(block, field="close"):
    # Reads one price from a Kalshi OHLC block, in dollars, without guessing units.
    block = block or {}
    if block.get(f"{field}_dollars") not in (None, ""):
        x = float(block[f"{field}_dollars"])
    elif block.get(field) not in (None, ""):
        v = block[field]
        # Integer = legacy cents format. Strings/floats are dollars.
        x = v / 100 if isinstance(v, int) else float(v)
    else:
        return np.nan
    if not 0.0 <= x <= 1.0:
        raise ValueError(f"Kalshi price {x} from {block} is outside [0, 1]")
    return x


def _kalshi_candles_to_df(candles):
    rows = []
    for c in candles:
        bid = _price_from_block(c.get("yes_bid"))
        ask = _price_from_block(c.get("yes_ask"))
        rows.append({
            "timestamp": pd.to_datetime(c["end_period_ts"], unit="s", utc=True),
            "kalshi_bid": bid,
            "kalshi_ask": ask,
            "kalshi_last": _price_from_block(c.get("price")),
            "kalshi_volume": pd.to_numeric(c.get("volume_fp", c.get("volume")), errors="coerce"),
        })
    return pd.DataFrame(rows)


def get_kalshi_candles(ticker, tier, start_utc, end_utc, period_minutes=BAR_MINUTES):
    # Downloads Kalshi candles in chunks of at most 5000 bars.
    # Returns (DataFrame, number of failed chunks).
    if tier == "live":
        series_ticker = ticker.split("-")[0]
        path = f"/series/{series_ticker}/markets/{ticker}/candlesticks"
    else:
        path = f"/historical/markets/{ticker}/candlesticks"

    start_ts, end_ts = int(start_utc.timestamp()), int(end_utc.timestamp())
    step = KALSHI_MAX_CANDLES_PER_REQUEST * period_minutes * 60

    frames, failed = [], 0
    cur = start_ts
    while cur < end_ts:
        c_end = min(cur + step - period_minutes * 60, end_ts)
        params = {"start_ts": cur, "end_ts": c_end, "period_interval": period_minutes}
        try:
            body = kalshi_get(path, params)
            frame = _kalshi_candles_to_df(body.get("candlesticks", []))
            if not frame.empty:
                frames.append(frame)
        except FetchError as e:
            failed += 1
            print(f"  !! Kalshi chunk failed [{cur}, {c_end}]: {e}")
        cur = c_end + period_minutes * 60
        time.sleep(0.1)  # be polite to the public API

    if not frames:
        return pd.DataFrame(columns=["timestamp", "kalshi_bid", "kalshi_ask", "kalshi_last", "kalshi_volume"]), failed

    out = (pd.concat(frames, ignore_index=True)
             .drop_duplicates("timestamp")
             .sort_values("timestamp")
             .reset_index(drop=True))
    return out, failed


def kalshi_midpoint(kalshi_df, max_spread=MAX_KALSHI_SPREAD):
    # Adds kalshi_spread and the midpoint price `kalshi`; drops bars with no
    # usable two-sided quote. Returns (DataFrame, number dropped).
    df = kalshi_df.copy()
    df["kalshi_spread"] = df["kalshi_ask"] - df["kalshi_bid"]
    ok = df["kalshi_bid"].notna() & df["kalshi_ask"].notna() & (df["kalshi_spread"] >= 0) & (df["kalshi_spread"] <= max_spread + 1e-9)
    df["kalshi"] = (df["kalshi_bid"] + df["kalshi_ask"]) / 2
    return df[ok].reset_index(drop=True), int((~ok).sum())

## Download: Polymarket price history

Uses the CLOB `prices-history` endpoint with `fidelity` = `BAR_MINUTES`.
(The `data-api.polymarket.com/v2` endpoint returns no data for older markets,
so it isn't used.) Windows longer than 14 days are downloaded in chunks.

In [ ]:
def _fetch_one_polymarket_chunk(token_id, c_start, c_end, fidelity_minutes):
    params = {
        "market": str(token_id),
        "startTs": int(c_start),
        "endTs": int(c_end),
        "fidelity": int(fidelity_minutes),
    }
    body = http_get_json(f"{POLY_CLOB_BASE}/prices-history", params)
    rows = body.get("history", [])
    if not rows:
        return pd.DataFrame(columns=["timestamp", "polymarket"])
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["t"], unit="s", utc=True)
    df["polymarket"] = pd.to_numeric(df["p"], errors="coerce")
    return df[["timestamp", "polymarket"]].dropna()


def get_polymarket_history(token_id, start_utc, end_utc, fidelity_minutes=BAR_MINUTES):
    # Returns (DataFrame, number of failed chunks, number of out-of-range prices dropped).
    start_ts, end_ts = int(start_utc.timestamp()), int(end_utc.timestamp())
    window = POLY_MAX_WINDOW_DAYS * 86400

    frames, failed = [], 0
    cur = start_ts
    while cur < end_ts:
        c_end = min(cur + window, end_ts)
        try:
            frame = _fetch_one_polymarket_chunk(token_id, cur, c_end, fidelity_minutes)
            if not frame.empty:
                frames.append(frame)
        except FetchError as e:
            failed += 1
            print(f"  !! Polymarket chunk failed [{cur}, {c_end}]: {e}")
        cur = c_end
        time.sleep(0.1)

    if not frames:
        return pd.DataFrame(columns=["timestamp", "polymarket"]), failed, 0

    out = (pd.concat(frames, ignore_index=True)
             .drop_duplicates("timestamp")
             .sort_values("timestamp")
             .reset_index(drop=True))
    in_range = out["polymarket"].between(0.0, 1.0)
    n_bad = int((~in_range).sum())
    if n_bad:
        print(f"  dropped {n_bad} Polymarket price(s) outside [0, 1]")
    return out[in_range].reset_index(drop=True), failed, n_bad

## Align: pair the two series

The time grid is Kalshi's bar ends. Each bar is paired with the **nearest**
Polymarket point within ±half a bar (±30 s for 1-minute bars). This keeps the
pairing unbiased (Polymarket's samples land a few seconds after each minute,
so "latest before" would make Polymarket look about a minute stale) while
limiting any look-ahead to half a bar. Each Polymarket point is used at most
once, nothing is filled in, and unmatched bars are dropped.

`dt_minutes` is the time since the previous row, since rows are not always
evenly spaced (minutes with no Kalshi quote update are missing).

In [ ]:
def align_and_compute_gap(kalshi_df, poly_df, bar_minutes=BAR_MINUTES):
    if kalshi_df.empty or poly_df.empty:
        return pd.DataFrame()

    half_bar = pd.Timedelta(minutes=bar_minutes) / 2
    k = kalshi_df.sort_values("timestamp")
    p = poly_df.sort_values("timestamp").rename(columns={"timestamp": "poly_time"})
    p["timestamp"] = p["poly_time"]

    merged = pd.merge_asof(k, p, on="timestamp", direction="nearest", tolerance=half_bar)
    merged = merged.dropna(subset=["kalshi", "polymarket"])

    # One Polymarket point per Kalshi bar and vice versa: if a Polymarket point
    # sits exactly between two bars, keep the closer pairing.
    merged["poly_offset_s"] = (merged["poly_time"] - merged["timestamp"]).dt.total_seconds()
    merged = (merged.assign(_abs=merged["poly_offset_s"].abs())
                    .sort_values(["poly_time", "_abs"])
                    .drop_duplicates("poly_time", keep="first")
                    .drop(columns="_abs")
                    .sort_values("timestamp")
                    .reset_index(drop=True))

    merged["gap"] = merged["kalshi"] - merged["polymarket"]
    merged["dt_minutes"] = merged["timestamp"].diff().dt.total_seconds() / 60
    return merged

## Flag outliers

For each point: how far is the gap from the median gap of the points within
`OUTLIER_WINDOW` around it, measured in MAD units? The MAD has a floor of half
a cent, so long flat stretches (MAD = 0) can still flag a spike. Points are
flagged, not deleted.

In [ ]:
def flag_gap_outliers(merged_df, window=OUTLIER_WINDOW, z_threshold=OUTLIER_Z, mad_floor=MAD_FLOOR):
    if merged_df.empty:
        return merged_df
    gap = merged_df.set_index("timestamp")["gap"]
    med = gap.rolling(window, center=True, min_periods=3).median()
    mad = (gap - med).abs().rolling(window, center=True, min_periods=3).median()
    z = 0.6745 * (gap - med) / np.maximum(mad.fillna(mad_floor), mad_floor)

    result = merged_df.copy()
    result["gap_zscore"] = z.to_numpy()
    result["is_outlier"] = (result["gap_zscore"].abs() > z_threshold).to_numpy()
    return result

## Plot

In [ ]:
def plot_gap(merged_df, output_path, title, show=True):
    fig, (price_ax, gap_ax) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    price_ax.plot(merged_df["timestamp"], merged_df["kalshi"], label="Kalshi (mid)", linewidth=0.8)
    price_ax.plot(merged_df["timestamp"], merged_df["polymarket"], label="Polymarket", linewidth=0.8)
    price_ax.set_ylabel("Implied probability")
    price_ax.set_title(title)
    price_ax.legend()

    gap_ax.plot(merged_df["timestamp"], merged_df["gap"], color="black", linewidth=0.6)
    gap_ax.axhline(0, color="gray", linewidth=0.8)
    flagged = merged_df[merged_df["is_outlier"]]
    if not flagged.empty:
        gap_ax.scatter(flagged["timestamp"], flagged["gap"], color="red", marker="x",
                       s=40, zorder=5, label="flagged outlier")
        gap_ax.legend()
    gap_ax.set_ylabel("Gap (Kalshi − Poly)")
    gap_ax.set_xlabel("Time (UTC)")

    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    if show:
        plt.show()
    plt.close(fig)

## Step 1 (optional) — look up tickers / tokens you don't have yet

These cells only print lookups; they aren't needed for the 9 events below,
whose IDs are already filled in.

In [ ]:
find_kalshi_markets("KXNEXTPOPE")

In [ ]:
print(resolve_polymarket_yes_token("world-series-champion-2025", "Los Angeles Dodgers"))
print(resolve_polymarket_yes_token("nba-champion-2024-2025", "Oklahoma City Thunder"))
print(resolve_polymarket_yes_token("oscars-best-picture", "Anora"))

## Step 2 — the 9 events

`start` and `end` are dates (UTC, `end` inclusive). The pull also stops at the
earlier of the two markets' close times, whichever comes first, so no
post-settlement quotes get in.

In [ ]:
EVENTS = [
    # --- Politics ---
    {
        "label": "election_2024_DJT_wins",
        "category": "Politics",
        "kalshi_ticker": "PRES-2024-DJT",
        "poly_token_id": "21742633143463906290569050155826241533067272736897614950488156847949938836455",
        "start": "2024-10-01", "end": "2024-11-06",
    },
    {
        "label": "fomc_sep2026_hike_25bp",
        "category": "Politics",
        "kalshi_ticker": "KXFEDDECISION-26SEP-H25",
        "poly_token_id": "63842529068710005716169325380315470359047749786610778647370693404952498013178",
        "start": "2026-08-15", "end": "2026-09-17",
    },
    {
        "label": "shutdown_oct2025_yes",
        "category": "Politics",
        "kalshi_ticker": "KXGOVSHUT-25OCT01",
        "poly_token_id": "98045093822431547938365423095786421804452886266888528428746910959667685407691",
        "start": "2025-09-01", "end": "2025-10-02",
    },
    # --- Sports ---
    {
        "label": "super_bowl_lx_seattle_wins",
        "category": "Sports",
        "kalshi_ticker": "KXSB-26-SEA",
        "poly_token_id": "91737931954079461205792748723730956466398437395923414328893692961489566016241",
        "start": "2026-01-01", "end": "2026-02-10",
    },
    {
        "label": "world_series_2025_dodgers_win",
        "category": "Sports",
        "kalshi_ticker": "KXMLB-25-LAD",
        "poly_token_id": "50056291296373013403053264672448796961558656386670931029485176637893947669174",
        "start": "2025-10-01", "end": "2025-11-05",
    },
    {
        "label": "nba_finals_2025_okc_win",
        "category": "Sports",
        "kalshi_ticker": "KXNBA-25-OKC",
        "poly_token_id": "83527644927648970835156950007024690327726158617181889316317174894904268227846",
        "start": "2025-05-15", "end": "2025-06-25",
    },
    # --- Culture ---
    {
        "label": "papal_conclave_2025_prevost",
        "category": "Culture",
        "kalshi_ticker": "KXNEXTPOPE-35-RPRE",
        "poly_token_id": "113845724700054083561790691604568760899627369924969834963221702983533500982036",
        "start": "2025-04-15", "end": "2025-05-10",
    },
    {
        "label": "oscars_2025_anora_best_pic",
        "category": "Culture",
        "kalshi_ticker": "KXOSCARPIC-25-A",
        "poly_token_id": "1018044885195187396947687732754809918865343613107923057688573272106246886316",
        "start": "2025-02-01", "end": "2025-03-03",
    },
    {
        "label": "taylor_swift_engagement_yes",
        "category": "Culture",
        "kalshi_ticker": "KXENGAGEMENTSWIFTKELCE-25",
        "poly_token_id": "65243545140060488194767449391633027630960354524222035359867983913384312052417",
        "start": "2025-07-15", "end": "2025-08-27",
    },
]

print(f"{len(EVENTS)} events configured.")

## Step 3 — run the full pull for every event

For each event: verify both markets, work out the cutoff, download both
series, take the Kalshi midpoint, align, flag outliers, save the CSV and plot.
If something goes wrong for one event, the error is printed and the run moves
on to the next event; the summary's `status` column says what happened.

At 1-minute bars this takes a few minutes in total.

In [ ]:
GAP_COLUMNS = [
    # same first six columns as the old gap.csv
    "timestamp", "polymarket", "kalshi", "gap", "gap_zscore", "is_outlier",
    # new
    "kalshi_bid", "kalshi_ask", "kalshi_spread", "kalshi_last", "kalshi_volume",
    "poly_offset_s", "dt_minutes",
]


def process_event(event):
    label = event["label"]
    row = {k: event[k] for k in ("label", "category", "start", "end", "kalshi_ticker", "poly_token_id")}

    # 1. Verify both markets and find their close times
    k_market, tier = get_kalshi_market(event["kalshi_ticker"])
    p_market = get_polymarket_market(event["poly_token_id"])
    print(f"  Kalshi:     {k_market.get('title')} [{k_market.get('yes_sub_title')}] ({tier} API)")
    print(f"  Polymarket: {p_market.get('question')} [Yes]")

    start_utc = to_utc(event["start"])
    end_utc = to_utc(event["end"]) + timedelta(days=1) - timedelta(seconds=1)
    k_close = parse_time(k_market.get("close_time"))
    p_close = parse_time(p_market.get("closedTime"))
    cutoff = min(t for t in (pd.Timestamp(end_utc), k_close, p_close) if t is not None)
    print(f"  window: {start_utc:%Y-%m-%d %H:%M} -> {cutoff:%Y-%m-%d %H:%M} UTC "
          f"(Kalshi close {k_close}, Polymarket close {p_close})")

    # 2. Download
    kalshi_raw, k_failed = get_kalshi_candles(event["kalshi_ticker"], tier, start_utc, cutoff.to_pydatetime())
    poly_df, p_failed, p_bad = get_polymarket_history(event["poly_token_id"], start_utc, cutoff.to_pydatetime())
    print(f"  downloaded {len(kalshi_raw)} Kalshi bars, {len(poly_df)} Polymarket points")

    # 3. Keep only data strictly before the cutoff (a Kalshi bar ending after
    #    the close contains the settlement book), then take the midpoint
    kalshi_raw = kalshi_raw[kalshi_raw["timestamp"] <= cutoff]
    poly_df = poly_df[poly_df["timestamp"] <= cutoff]
    kalshi_df, n_wide = kalshi_midpoint(kalshi_raw)
    print(f"  dropped {n_wide} Kalshi bars with no quote or spread > {MAX_KALSHI_SPREAD}")

    # 4. Align, flag, save
    merged = align_and_compute_gap(kalshi_df, poly_df)
    if merged.empty:
        raise FetchError("no aligned observations")
    merged = flag_gap_outliers(merged)[GAP_COLUMNS]

    out_dir = OUT_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_dir / "gap.csv", index=False)
    plot_gap(merged, out_dir / "gap_plot.png", title=label)

    gap = merged["gap"]
    row.update({
        "n_points": len(merged),
        "mean_gap": gap.mean(),
        "mean_abs_gap": gap.abs().mean(),
        "median_abs_gap": gap.abs().median(),
        "max_abs_gap": gap.abs().max(),
        "gap_std": gap.std(),
        # new columns
        "n_outliers": int(merged["is_outlier"].sum()),
        "max_abs_gap_excl_outliers": gap[~merged["is_outlier"]].abs().max(),
        "bar_minutes": BAR_MINUTES,
        "cutoff_utc": str(cutoff),
        "kalshi_bars_downloaded": len(kalshi_raw),
        "kalshi_bars_dropped_spread": n_wide,
        "poly_points_downloaded": len(poly_df),
        "poly_points_dropped_range": p_bad,
        "failed_chunks": k_failed + p_failed,
        "median_dt_minutes": merged["dt_minutes"].median(),
        "max_dt_minutes": merged["dt_minutes"].max(),
        "kalshi_total_volume": merged["kalshi_volume"].sum(),
        "kalshi_title": f"{k_market.get('title')} [{k_market.get('yes_sub_title')}]",
        "poly_question": p_market.get("question"),
        "status": "ok" if (k_failed + p_failed) == 0 else "INCOMPLETE: some chunks failed",
    })
    print(f"  done: {len(merged)} matched points, {row['n_outliers']} flagged, "
          f"mean |gap| {row['mean_abs_gap']:.4f}, max |gap| {row['max_abs_gap']:.4f}")
    return row


summary_rows = []
for event in EVENTS:
    print(f"\n=== {event['label']} ({event['category']}) ===")
    if "FILL_IN" in (event.get("kalshi_ticker"), event.get("poly_token_id")):
        print("  SKIPPED: missing Kalshi ticker or Polymarket token ID")
        continue
    try:
        summary_rows.append(process_event(event))
    except (FetchError, ValueError) as e:
        print(f"  FAILED: {e}")
        summary_rows.append({**{k: event[k] for k in ("label", "category", "start", "end",
                                                      "kalshi_ticker", "poly_token_id")},
                             "status": f"FAILED: {e}"})

## Step 4 — summary table across all events

One row per event, saved as `out/summary.csv`. The first twelve columns match
the old summary; the rest are new checks (cutoff time, how many bars were
dropped, failed chunks, the time spacing, and the market titles so you can see
both sides are the same question).

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "summary.csv", index=False)

bad = summary_df[summary_df["status"] != "ok"]
if not bad.empty:
    print("Events that need attention:")
    print(bad[["label", "status"]].to_string(index=False))

summary_df